In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Define the three conditions with different difficulty profiles
conditions = {
    'Condition_A': {  # Easy condition
        'mental_demand_mean': 2.5,
        'success_mean': 5.8,
        'frustration_mean': 2.2,
        'trajectory_ease_mean': 5.5,
        'option_clarity_mean': 5.7,
        'preference_learning_mean': 5.9
    },
    'Condition_B': {  # Medium condition
        'mental_demand_mean': 4.2,
        'success_mean': 4.1,
        'frustration_mean': 3.8,
        'trajectory_ease_mean': 4.0,
        'option_clarity_mean': 4.3,
        'preference_learning_mean': 4.5
    },
    'Condition_C': {  # Hard condition
        'mental_demand_mean': 5.8,
        'success_mean': 2.9,
        'frustration_mean': 5.5,
        'trajectory_ease_mean': 2.8,
        'option_clarity_mean': 3.1,
        'preference_learning_mean': 3.2
    }
}

# Number of participants (each will appear once per condition)
n_participants = 30

def generate_correlated_responses(base_mental_demand, base_success, base_frustration, 
                                base_traj_ease, base_opt_clarity, base_pref_learn, 
                                correlation_strength=0.6):
    """Generate correlated survey responses respecting inverse relationships"""
    
    # Generate base mental demand score
    mental_demand = np.clip(np.random.normal(base_mental_demand, 1.0), 1, 7)
    
    # Success is inversely related to mental demand and frustration
    success_noise = np.random.normal(0, 0.8)
    success = base_success - correlation_strength * (mental_demand - 4) + success_noise
    success = np.clip(success, 1, 7)
    
    # Frustration is positively related to mental demand
    frustration_noise = np.random.normal(0, 0.8)
    frustration = base_frustration + correlation_strength * (mental_demand - 4) + frustration_noise
    frustration = np.clip(frustration, 1, 7)
    
    # Other questions have some correlation with overall difficulty
    difficulty_factor = (mental_demand + frustration - success) / 3 - 4
    
    traj_ease = base_traj_ease - 0.2 * difficulty_factor + np.random.normal(0, 0.8)
    traj_ease = np.clip(traj_ease, 1, 7)
    
    opt_clarity = base_opt_clarity - 0.1 * difficulty_factor + np.random.normal(0, 0.8)
    opt_clarity = np.clip(opt_clarity, 1, 7)
    
    pref_learn = base_pref_learn - 0.3 * difficulty_factor + np.random.normal(0, 0.9)
    pref_learn = np.clip(pref_learn, 1, 7)
    
    return mental_demand, success, frustration, traj_ease, opt_clarity, pref_learn

# -------------------
# Response-style biases
# -------------------

bias_probabilities = {
    'none': 0.35,
    'acquiescence': 0.12,          # yea-saying
    'disacquiescence': 0.07,       # nay-saying
    'extreme': 0.12,               # endpoints
    'midpoint': 0.12,              # prefers middle
    'straightlining': 0.10,        # same answer
    'random': 0.06,                # random/noise
    'socially_desirable': 0.06     # up good, down bad
}

bias_types = list(bias_probabilities.keys())
bias_weights = np.array(list(bias_probabilities.values()))
bias_weights = bias_weights / bias_weights.sum()

POSITIVE_ITEMS_IDX = [1, 3, 4, 5]  # success, trajectory ease, option clarity, preference learning
NEGATIVE_ITEMS_IDX = [0, 2]        # mental demand, frustration

def sample_bias():
    return np.random.choice(bias_types, p=bias_weights)

def apply_bias(scores, bias_type):
    scores = np.array(scores, dtype=float)
    if bias_type == 'none':
        biased = scores
    elif bias_type == 'acquiescence':
        shift = np.random.binomial(1, 0.7, size=scores.shape)
        biased = scores + shift
    elif bias_type == 'disacquiescence':
        shift = np.random.binomial(1, 0.7, size=scores.shape)
        biased = scores - shift
    elif bias_type == 'extreme':
        biased = []
        for s in scores:
            if s >= 4:
                biased.append(7 if np.random.rand() < 0.8 else 6)
            else:
                biased.append(1 if np.random.rand() < 0.8 else 2)
        biased = np.array(biased, dtype=float)
    elif bias_type == 'midpoint':
        biased = []
        for s in scores:
            r = np.random.rand()
            if r < 0.75:
                biased.append(4)
            elif r < 0.9:
                biased.append(np.random.choice([3, 5]))
            else:
                biased.append(np.rint(s))
        biased = np.array(biased, dtype=float)
    elif bias_type == 'straightlining':
        anchor = int(np.clip(np.rint(np.mean(scores) + np.random.normal(0, 0.3)), 1, 7))
        biased = np.full_like(scores, anchor, dtype=float)
    elif bias_type == 'random':
        biased = np.random.randint(1, 8, size=scores.shape).astype(float)
    elif bias_type == 'socially_desirable':
        biased = scores.copy()
        for idx in POSITIVE_ITEMS_IDX:
            if np.random.rand() < 0.65:
                biased[idx] += 1
        for idx in NEGATIVE_ITEMS_IDX:
            if np.random.rand() < 0.7:
                biased[idx] -= 1
    else:
        biased = scores

    biased = np.clip(biased + np.random.normal(0, 0.15, size=len(biased)), 1, 7)
    return biased

# Generate data: each participant appears once per condition, bias held constant per participant
data = []
condition_names = list(conditions.keys())

for pid in range(1, n_participants + 1):
    respondent_bias = sample_bias()
    for condition_name in condition_names:
        params = conditions[condition_name]
        base = generate_correlated_responses(
            params['mental_demand_mean'],
            params['success_mean'], 
            params['frustration_mean'],
            params['trajectory_ease_mean'],
            params['option_clarity_mean'],
            params['preference_learning_mean']
        )
        biased = apply_bias(base, respondent_bias)
        biased = np.rint(np.clip(biased, 1, 7)).astype(int)
        data.append({
            'participant_id': pid,
            'condition': condition_name,
            'Q1_mental_demand': int(biased[0]),
            'Q2_success': int(biased[1]),
            'Q3_frustration': int(biased[2]),
            'Q4_trajectory_ease': int(biased[3]),
            'Q5_option_clarity': int(biased[4]),
            'Q6_preference_learning': int(biased[5])
        })

# Create DataFrame
df = pd.DataFrame(data)

# Display summary statistics
print("=== SUMMARY STATISTICS BY CONDITION ===")
print(df.groupby('condition')[['Q1_mental_demand', 'Q2_success', 'Q3_frustration', 
                              'Q4_trajectory_ease', 'Q5_option_clarity', 'Q6_preference_learning']].mean().round(2))

print("\n=== STANDARD DEVIATIONS BY CONDITION ===")
print(df.groupby('condition')[['Q1_mental_demand', 'Q2_success', 'Q3_frustration', 
                              'Q4_trajectory_ease', 'Q5_option_clarity', 'Q6_preference_learning']].std().round(2))

# Save data to CSV
df.to_csv('fake_user_study_survey_data.csv', index=False)
print(f"\nData saved to 'fake_user_study_survey_data.csv'")
print(f"Unique participants: {n_participants}")
print(f"Rows (participants × conditions): {len(df)}")
print(f"Conditions per participant: {len(conditions)}")

=== SUMMARY STATISTICS BY CONDITION ===
             Q1_mental_demand  Q2_success  Q3_frustration  Q4_trajectory_ease  \
condition                                                                       
Condition_A               2.9        5.77            2.40                5.67   
Condition_B               4.4        3.80            4.30                4.27   
Condition_C               5.3        2.57            5.73                3.33   

             Q5_option_clarity  Q6_preference_learning  
condition                                               
Condition_A               5.47                    6.03  
Condition_B               4.50                    5.03  
Condition_C               3.37                    3.43  

=== STANDARD DEVIATIONS BY CONDITION ===
             Q1_mental_demand  Q2_success  Q3_frustration  Q4_trajectory_ease  \
condition                                                                       
Condition_A              1.24        1.04            1.63        